In [27]:
### import libraries
import json, tarfile
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
import shutil
import urllib.request
import re


import os

In [28]:
### set seed for reproducibility

SEED = 37

In [29]:
### configure settings for file import and image extraction
# set/create data directory
DATA_ROOT = Path("./data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# set subfolders / paths for files
TAR_PATH = DATA_ROOT / "gzgc.coco.tar.gz"
DATA_DIR = DATA_ROOT / "gzgc_dataset"
DONE_FLAG = DATA_DIR / ".extracted_ok"

JSON_NAME  = None                 # None = auto-detect the first .json in the file
IMG_ROOT   = Path("images")       # folder containing the files named in coco["images"]
OUT_DIR_DEV    = Path("./data/gzgc_dataset/dev_crops")
OUT_DIR_DEV.mkdir(parents=True, exist_ok=True)
OUT_DIR_EVAL    = Path("./data/gzgc_dataset/eval_crops")
OUT_DIR_EVAL.mkdir(parents=True, exist_ok=True)

# set parameters for cropping
PAD, OUT_SIZE, MIN_SIDE = 0.10, 224, 32

In [30]:
### download dataset

### URLs where the data set is available to be downloaded from
URLS = [
    "https://storage.googleapis.com/public-datasets-lila/wild-me/gzgc.coco.tar.gz",
    "https://lilawildlife.blob.core.windows.net/lila-wildlife/wild-me/gzgc.coco.tar.gz",
    "http://us-west-2.opendata.source.coop.s3.amazonaws.com/agentmorris/lila-wildlife/wild-me/gzgc.coco.tar.gz",
]

### if not already there, download data file
if not TAR_PATH.exists():
    # download to a temp name so a failed run isn't mistaken for a complete file
    tmp = TAR_PATH.with_suffix(".part")
    for url in URLS:
        try:
            print("Downloading from", url)
            urllib.request.urlretrieve(url, tmp)
            tmp.rename(TAR_PATH)
            break
        except Exception as e:
            print("Failed:", e)
            tmp.unlink(missing_ok=True)
    else:
        raise RuntimeError("All download mirrors failed")
else:
    print("Archive already downloaded:", TAR_PATH)


Archive already downloaded: data\gzgc.coco.tar.gz


In [31]:
### extract from .gz file

if not DONE_FLAG.exists():
    with tarfile.open(TAR_PATH, "r:*") as t:
        t.extractall(DATA_DIR, filter="data")    # drop filter= on Python < 3.12 unless your patch release has the backport
    DONE_FLAG.touch()
else:
    print("Dataset already extracted:", DATA_DIR)

Dataset already extracted: data\gzgc_dataset


In [32]:
### list and select the correct json file

'''per the data set writeup:
"We have collapsed the entire dataset into a single “train” label and have left “val” and “test” empty;"
https://lila.science/datasets/great-zebra-giraffe-id
'''
json_files = sorted(DATA_DIR.rglob("*.json"))

### list json files
print(json_files)

# select the train set (per data set specifications)
coco = json.loads(json_files[1].read_text())


[WindowsPath('data/gzgc_dataset/gzgc.coco/annotations/instances_test2020.json'), WindowsPath('data/gzgc_dataset/gzgc.coco/annotations/instances_train2020.json'), WindowsPath('data/gzgc_dataset/gzgc.coco/annotations/instances_val2020.json')]


In [33]:
### check image file path and update it as required
# image file name
sample = coco["images"][0]["file_name"]
# image file path
matches = list(DATA_DIR.rglob(Path(sample).name))

# print file path
print(sample, "->", matches[:1])

# update image root path to match path above
IMG_ROOT = DATA_DIR/"gzgc.coco/images/train2020"

# print image path to verify
print(IMG_ROOT)

000000000001.jpg -> [WindowsPath('data/gzgc_dataset/gzgc.coco/images/train2020/000000000001.jpg')]
data\gzgc_dataset\gzgc.coco\images\train2020


In [34]:
### import compressed data file (coco.tar format) and examine contents

with tarfile.open(TAR_PATH, "r:*") as t:   # "r:*" handles plain, .gz, .bz2, .xz
    members = [m for m in t.getmembers() if m.isfile() and m.name.endswith(".json")]
    print([m.name for m in members])       # see what's inside
    member = next(m for m in members if JSON_NAME in (None, m.name))
    coco = json.load(t.extractfile(member))

images = {im["id"]: im for im in coco["images"]}
cats   = {c["id"]: c["name"] for c in coco["categories"]}

['gzgc.coco/annotations/instances_train2020.json', 'gzgc.coco/annotations/instances_val2020.json', 'gzgc.coco/annotations/instances_test2020.json']


In [35]:
### inspect first line of annotations to determine where the animal ID is located

print(coco["annotations"][0])



{'bbox': [895.5, 437.0, 1221.0, 690.0], 'theta': 0.0, 'viewpoint': 'left', 'segmentation': [[896, 437, 2116, 437, 2116, 1127, 896, 1127, 896, 437]], 'segmentation_bbox': [896, 437, 1220, 690], 'area': 841800, 'iscrowd': 0, 'id': 1, 'image_id': 1, 'category_id': 1, 'uuid': '7cfac6bc-379a-4859-a5c8-b21b06d2fe3d', 'individual_ids': [2, 1, 3, 3459], 'isinterest': 0, 'name': 'IBEIS_PZ_1561', 'review_ids': []}


In [36]:
### function to extract animal ID and group images of the same animal together to prevent leakage
def get_animal_id(ann):
    name = ann.get("name")
    if name and not str(name).startswith("unk"):
        return name
    return f"unk_img{ann['image_id']}"              # unnamed: group by image so crops of one photo stay together

In [37]:
### group annotations by source image
by_image = {}
for ann in coco["annotations"]:
    if ann.get("iscrowd", 0):
        continue
    # if the dataset includes rotated boxes, uncomment to skip them:
    if ann.get("theta", 0) != 0:
         continue
    by_image.setdefault(ann["image_id"], []).append(ann)

In [38]:
IMG_ROOT

WindowsPath('data/gzgc_dataset/gzgc.coco/images/train2020')

In [39]:
### crop images based on coordinates provided in the annotation file

#OUT_DIR.mkdir(parents=True, exist_ok=True)      # single flat folder, no per-class subfolders

def safe(s):
    return re.sub(r"[^A-Za-z0-9_-]+", "-", str(s))

from collections import Counter

#OUT_DIR_LIST = [OUT_DIR_DEV, OUT_DIR_EVAL]

#for OUT_DIR_INC in OUT_DIR_LIST:

OUT_DIR = OUT_DIR_DEV / "cropped_resized"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR.resolve())
print("images with annotations:", len(by_image))
print("PAD, OUT_SIZE, MIN_SIDE:", PAD, OUT_SIZE, MIN_SIDE)
print("IMG_ROOT:", IMG_ROOT.resolve(), "| exists:", IMG_ROOT.exists())

skips = Counter()

records = []
for img_id, anns in by_image.items():
    info = images[img_id]
    path = IMG_ROOT / info["file_name"]
    if not path.exists():
        skips["missing_image"] += 1
        if skips["missing_image"] <= 3:
            print("missing image:", path)
        continue
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    W, H = img.size

    for ann in anns:
        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue
        px, py = w * PAD, h * PAD
        x1, y1 = max(0, int(x - px)), max(0, int(y - py))
        x2, y2 = min(W, int(x + w + px)), min(H, int(y + h + py))
        if min(x2 - x1, y2 - y1) < MIN_SIDE:
            continue

        crop = ImageOps.pad(img.crop((x1, y1, x2, y2)), (OUT_SIZE, OUT_SIZE), color=(0, 0, 0))
        label = cats[ann["category_id"]]

        # annotation id is unique, so filenames can't collide even in one flat folder
        fname = f"ann{ann['id']}_img{img_id}_{safe(label)}.jpg"
        crop.save(OUT_DIR / fname, quality=95)
        skips["saved"] += 1

        if label == 'giraffe_masai':
            classif = 0
        elif label == 'zebra_plains':
            classif = 1
        else:
            classif = 10

        records.append(dict(
            crop=fname, path=str(OUT_DIR / fname),
            image_id=img_id, ann_id=ann["id"],
            animal_id=get_animal_id(ann), viewpoint=ann.get("viewpoint"),
            isinterest=ann.get("isinterest"), label=label,
            classification= classif,
            box_w=w, box_h=h, img_w=W, img_h=H,
        ))
print(dict(skips))
df = pd.DataFrame(records)
df.to_csv("./data/gzgc_dataset/annot_all_crops.csv", index=False)
print(len(df), "crops;", len(list(OUT_DIR.glob("*.jpg"))), "files on disk")

OUT_DIR: C:\Users\eenoo\Downloads\STAT_5720_Projects\MegaDetector-Classifier\data\gzgc_dataset\dev_crops\cropped_resized
images with annotations: 4619
PAD, OUT_SIZE, MIN_SIDE: 0.1 224 32
IMG_ROOT: C:\Users\eenoo\Downloads\STAT_5720_Projects\MegaDetector-Classifier\data\gzgc_dataset\gzgc.coco\images\train2020 | exists: True
{'saved': 6411}
6411 crops; 6411 files on disk


In [40]:
len(records)


6411

In [41]:
### export image meta data into dataframe and then csv


print(df.label.value_counts())
print(df.groupby("label")[["box_w", "box_h"]].describe().T)   # crop size by class
print(pd.crosstab(df.label, df.viewpoint))                     # viewpoint by class


label
zebra_plains     5811
giraffe_masai     600
Name: count, dtype: int64
label        giraffe_masai  zebra_plains
box_w count     600.000000   5811.000000
      mean     1035.689261   1038.946998
      std       565.357345    603.258942
      min       114.832536     73.564593
      25%       553.031250    565.500000
      50%       995.931278    869.382022
      75%      1394.822917   1405.671494
      max      2991.000000   2987.012987
box_h count     600.000000   5811.000000
      mean     1127.978696    713.549930
      std       565.588581    390.529087
      min       123.803828     94.198565
      25%       629.915730    409.855130
      50%      1129.050926    609.550562
      75%      1585.674157    959.415584
      max      2985.351562   2241.210938
viewpoint      back  backleft  backright  front  frontleft  frontright  left  \
label                                                                          
giraffe_masai     9        49          7      5         64         

In [42]:
### dataset splitting to reserve ~ half of the images to do other evaluation with the fine-tuned model
### stratify the images by classification (animal) and group by animal ID to avoid leakage

sgkf = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
dev_idx, holdout_idx = next(sgkf.split(df, df.label, groups=df.animal_id))

df["split"] = "holdout"
df.loc[df.index[dev_idx], "split"] = "dev"

In [43]:
### extract validation set (0.2) from the training half of the dataset

dev = df[df.split == "dev"]
sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf2.split(dev, dev.label, groups=dev.animal_id))
df.loc[dev.index[val_idx], "split"] = "val"
df.loc[dev.index[train_idx], "split"] = "train"

In [44]:
### data checks

# check no individual appears in both splits
assert df.groupby("animal_id").split.nunique().max() == 1

# check share of each class per split
ct = pd.crosstab(df.split, df.label)
print(ct)
print(ct.div(ct.sum(axis=0), axis=1).round(3))


label    giraffe_masai  zebra_plains
split                               
holdout            300          2906
train              240          2324
val                 60           581
label    giraffe_masai  zebra_plains
split                               
holdout            0.5           0.5
train              0.4           0.4
val                0.1           0.1


In [45]:
### log image data from dataframe to csv files for both sets of data

df.to_csv("./data/gzgc_dataset/dev_crops/annot_dev_crops.csv", index=False)
df[df.split == "holdout"].to_csv("./data/gzgc_dataset/eval_crops/annot_eval_crops.csv", index=False)

In [46]:
OUT_DIR_EVAL

WindowsPath('data/gzgc_dataset/eval_crops')

In [47]:
OUT_DIR_DEV

WindowsPath('data/gzgc_dataset/dev_crops')

In [48]:
### move eval data set to separate folder

assert "split" in df.columns, "Run the split cell first"

EVAL_DIR = OUT_DIR_EVAL/"cropped_resized"
OUT_DIR = OUT_DIR_DEV/"cropped_resized"
EVAL_DIR.mkdir(exist_ok=True)

for fname in df.loc[df["split"] == "holdout", "crop"]:
    src = OUT_DIR / fname
    if src.exists():                              # safe to rerun
        shutil.move(src, EVAL_DIR / fname)

df["path"] = [str((EVAL_DIR if s == "holdout" else OUT_DIR) / c)
              for s, c in zip(df["split"], df["crop"])]
df.to_csv("crops_index.csv", index=False)

df[df.split == "holdout"].to_csv("./data/gzgc_dataset/eval_crops/annot_eval_crops.csv", index=False)
df[df.split != "holdout"].to_csv("./data/gzgc_dataset/dev_crops/annot_dev_crops.csv", index=False)

In [49]:
### pull dev_crops
dev_files = {p.name for p in OUT_DIR.glob("*.jpg")} # was OUT_DIR_DEV
dev = df[df["crop"].isin(dev_files)].copy()
print("dev crops:", len(dev), "| holdout crops (left alone):", len(df) - len(dev))

dev crops: 3205 | holdout crops (left alone): 3206


In [50]:
### Split dev_crops data into train, test, and val sets stratified by class, grouped by animal ID

### Test data from dev_crops
# Test = 1 of 5 folds (~20% of dev), stratified by class, grouped by animal ID
sgkf_a = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_pos, test_pos = next(sgkf_a.split(dev, dev["label"], groups=dev["animal_id"]))
trainval = dev.iloc[trainval_pos]

### Val data from dev_crops
# Val = 1 of 4 folds of what remains (~20% of dev); the rest is train (~60%)
sgkf_b = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
train_pos, val_pos = next(sgkf_b.split(trainval, trainval["label"], groups=trainval["animal_id"]))

### Train data is remaining from dev_crops
# Train (~60% of dev)
dev["split"] = "train"
dev.loc[trainval.index[val_pos], "split"] = "val"
dev.loc[dev.index[test_pos], "split"] = "test"

# Checks: no animal in two splits, all three splits non-empty, class mix per split
assert dev.groupby("animal_id")["split"].nunique().max() == 1
assert set(dev["split"]) == {"train", "val", "test"}, set(dev["split"])
ct = pd.crosstab(dev["split"], dev["label"])
print(ct)
print(ct.div(ct.sum(axis=0), axis=1).round(3))

label  giraffe_masai  zebra_plains
split                             
test              60           581
train            180          1743
val               60           581
label  giraffe_masai  zebra_plains
split                             
test             0.2           0.2
train            0.6           0.6
val              0.2           0.2


In [51]:
# Write the full dev annotation file and the three files the training code expects.
# 'path' is just the filename: all dev crops sit flat in dev_crops, next to the CSVs.

dev.to_csv(OUT_DIR_DEV / "annot_dev_crops.csv", index=False)
for s in ["train", "val", "test"]:
    out = dev[dev["split"] == s].copy()
    out["path"] = out["crop"]
    out.to_csv(OUT_DIR / f"{s}_annotations_cropped.csv", index=False)
    print(s, out.shape)

train (1923, 14)
val (641, 14)
test (641, 14)


In [52]:
### Write markers to each image directory to tell the detector code that cropping has already been completed
mode_tag = 'trainval'
marker1 = os.path.join(OUT_DIR, f".crop_done_{mode_tag}")
os.makedirs(OUT_DIR, exist_ok=True)
open(marker1, 'w').close()

marker2 = os.path.join(EVAL_DIR, f".crop_done_{mode_tag}")
os.makedirs(EVAL_DIR, exist_ok=True)
open(marker2, 'w').close()